# 02 — Sessionization + time split

Raw events → chuỗi product theo session, chia train/val/test theo thời gian.

Quy tắc:
- Split theo thời gian bắt đầu session: train ≤ 16/11, val 17–23/11, test 24–30/11. Session vắt ngang mốc → bỏ (chống leakage).
- Trong session: khử duplicate liên tiếp cùng product; giữ session ≥ 2 products; cắt tối đa 100.
- Output: `processed/sessions_{train,val,test}/part_*.parquet` — cột `user_session, user_id, start_time, seq, n_items`.

In [1]:
import os
os.environ["POLARS_MAX_THREADS"] = "4"

from pathlib import Path
from datetime import datetime
import time
import polars as pl

DATASET_DIR = Path("/mnt/d/VSF/VSF-MiniApp-Ecommerce/ai-recommendation/dataset")
PARQUET_DIR = DATASET_DIR / "parquet"
PROCESSED_DIR = DATASET_DIR / "processed"
SPLITS = ["train", "val", "test"]

VAL_START = datetime(2019, 11, 17)
TEST_START = datetime(2019, 11, 24)
MIN_ITEMS = 2
MAX_ITEMS = 100
N_BUCKETS = 16

scan = pl.scan_parquet(PARQUET_DIR / "*.parquet")
print("OK")

OK


In [2]:
for s in SPLITS:
    (PROCESSED_DIR / f"sessions_{s}").mkdir(parents=True, exist_ok=True)

done = all(any((PROCESSED_DIR / f"sessions_{s}").glob("part_*.parquet")) for s in SPLITS)
if done:
    print("Đã có output — bỏ qua (xóa các thư mục sessions_* để build lại)")
else:
    t_all = time.time()
    for b in range(N_BUCKETS):
        t0 = time.time()
        ev = (scan
            .filter(pl.col("user_session").is_not_null() & pl.col("product_id").is_not_null())
            .filter(pl.col("user_session").hash() % N_BUCKETS == b)
            .select("user_session", "user_id", "product_id", "event_time")
            .unique()
            .collect(engine="streaming")
            .sort("user_session", "event_time"))

        split = (ev.group_by("user_session")
            .agg(pl.col("event_time").min().alias("start"), pl.col("event_time").max().alias("end"))
            .with_columns(
                pl.when(pl.col("end") < VAL_START).then(pl.lit("train"))
                  .when((pl.col("start") >= VAL_START) & (pl.col("end") < TEST_START)).then(pl.lit("val"))
                  .when(pl.col("start") >= TEST_START).then(pl.lit("test"))
                  .otherwise(pl.lit("drop"))  # session vắt ngang mốc
                  .alias("split"))
            .select("user_session", "split"))

        prev = pl.col("product_id").shift(1).over("user_session")
        seqs = (ev
            .join(split, on="user_session")
            .filter(pl.col("split") != "drop")
            .filter((pl.col("product_id") != prev) | prev.is_null())
            .group_by("user_session", "split")
            .agg(
                pl.col("user_id").first(),
                pl.col("event_time").first().alias("start_time"),
                pl.col("product_id").alias("seq"),
            )
            .filter(pl.col("seq").list.len() >= MIN_ITEMS)
            .with_columns(pl.col("seq").list.head(MAX_ITEMS))
            .with_columns(pl.col("seq").list.len().alias("n_items")))

        for s in SPLITS:
            (seqs.filter(pl.col("split") == s).drop("split")
                 .write_parquet(PROCESSED_DIR / f"sessions_{s}" / f"part_{b:02d}.parquet"))
        print(f"bucket {b+1}/{N_BUCKETS}: {ev.height:,} events → {seqs.height:,} sessions ({time.time()-t0:.0f}s)")
    print(f"Tổng: {time.time()-t_all:.0f}s")

bucket 1/16: 6,859,183 events → 705,048 sessions (37s)


bucket 2/16: 6,857,311 events → 703,620 sessions (13s)


bucket 3/16: 6,847,447 events → 703,515 sessions (13s)


bucket 4/16: 6,876,444 events → 706,336 sessions (13s)


bucket 5/16: 6,854,855 events → 704,530 sessions (13s)


...

bucket 12/16: 6,862,914 events → 704,351 sessions (14s)


bucket 13/16: 6,852,386 events → 703,831 sessions (14s)


bucket 14/16: 6,854,698 events → 704,947 sessions (13s)


bucket 15/16: 6,856,242 events → 704,570 sessions (12s)


bucket 16/16: 6,865,168 events → 705,258 sessions (12s)
Tổng: 229s


## Kiểm tra kết quả

In [3]:
import pandas as pd

stats = []
for s in SPLITS:
    d = (pl.scan_parquet(PROCESSED_DIR / f"sessions_{s}" / "*.parquet")
           .select(
               pl.len().alias("sessions"),
               pl.col("start_time").min().alias("first_session"),
               pl.col("start_time").max().alias("last_session"),
               pl.col("n_items").mean().round(1).alias("avg_items"),
           )
           .collect(engine="streaming")
           .with_columns(pl.lit(s).alias("split")))
    stats.append(d)
pl.concat(stats).select("split", "sessions", "first_session", "last_session", "avg_items").to_pandas()

,split,sessions,first_session,last_session,avg_items
0,train,8469647,2019-10-01 00:00:01,2019-11-16 23:59:31,5.8
1,val,1603647,2019-11-17 00:00:01,2019-11-23 23:58:51,5.9
2,test,1205598,2019-11-24 00:00:02,2019-11-30 23:59:32,5.5


In [4]:
# Cold-start: % target (item cuối) của val/test chưa từng xuất hiện trong train
train_vocab = (pl.scan_parquet(PROCESSED_DIR / "sessions_train" / "*.parquet")
                 .select(pl.col("seq").explode().unique().alias("product_id"))
                 .collect(engine="streaming"))
print(f"train vocab: {train_vocab.height:,} products")

for s in ["val", "test"]:
    targets = (pl.scan_parquet(PROCESSED_DIR / f"sessions_{s}" / "*.parquet")
                 .select(pl.col("seq").list.last().alias("product_id"))
                 .collect(engine="streaming"))
    cold = targets.join(train_vocab, on="product_id", how="anti").height
    print(f"{s}: {targets.height:,} sessions, target cold-start {cold/targets.height*100:.2f}%")

/tmp/ipykernel_18089/1005800989.py:3: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .select(pl.col("seq").explode().unique().alias("product_id"))


train vocab: 187,916 products
val: 1,603,647 sessions, target cold-start 0.67%


test: 1,205,598 sessions, target cold-start 2.81%


In [5]:
# Xem 3 session mẫu
(pl.scan_parquet(PROCESSED_DIR / "sessions_train" / "*.parquet")
   .select("user_session", "n_items", pl.col("seq").list.head(8))
   .head(3)
   .collect(engine="streaming")
   .to_pandas())

,user_session,n_items,seq
0,b2d0a1cd-6ba7-4cea-b00d-3ccb64107e4a,2,"[3701135, 3700779]"
1,c32e25b3-d591-40fe-8513-ce95daea1e31,3,"[1801995, 1801699, 1801940]"
2,818b749e-3dd5-4166-936f-2de6d6dab87e,2,"[1004870, 1004872]"


## Kết quả

Train 8.47M / val 1.60M / test 1.21M sessions · vocab 187,916 products · cold-start target val 0.67%, test 2.81%.